# 🎬 SCAIL-2 Character Animation on Colab
**End-to-end controlled character animation** — just feed a reference image + driving video!

> 基於 [zai-org/SCAIL-2](https://github.com/zai-org/SCAIL-2) · MIT License
> 使用 Wan 框架推理，SCAIL-2 內建 VAE + T5

**硬體需求**: T4 16GB+ (建議 A100 40GB 全速)
**時間**: 約 5-15 分鐘/段

## 📦 Step 1: 環境設置

In [ ]:
# @title Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q opencv-python imageio imageio-ffmpeg pillow numpy
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers accelerate safetensors
print('✅ Dependencies installed')

## 📥 Step 2: 下載 SCAIL-2 模型

In [ ]:
# @title Clone SCAIL-2 repo + download model (~14GB)
import os
os.environ['GIT_LFS_SKIP_SMUDGE'] = '1'

!git clone https://github.com/zai-org/SCAIL-2.git /content/SCAIL-2
%cd /content/SCAIL-2
!git checkout wan-scail2
!git submodule update --init --recursive

# Download model weights from HuggingFace
!pip install -q huggingface_hub
from huggingface_hub import snapshot_download
snapshot_download('zai-org/SCAIL-2', local_dir='/content/SCAIL-2/checkpoints', 
                  local_dir_use_symlinks=False, resume_download=True)
print('✅ Model downloaded')

## 🎯 Step 3: 上傳你的素材
上傳一張 **參考圖 (ref.jpg)** 和一個 **驅動影片 (driving.mp4)**

In [ ]:
# @title Upload reference image + driving video
from google.colab import files
import os

os.makedirs('/content/examples/my_char', exist_ok=True)

print('📷 上傳參考圖 (ref.jpg):')
uploaded = files.upload()
for name in uploaded:
    if name.lower().endswith(('.jpg','.jpeg','.png','.webp')):
        !mv "{name}" /content/examples/my_char/ref.jpg
        print(f'✅ 參考圖: {name}')

print('\n🎥 上傳驅動影片 (driving.mp4):')
uploaded = files.upload()
for name in uploaded:
    if name.lower().endswith('.mp4'):
        !mv "{name}" /content/examples/my_char/driving.mp4
        print(f'✅ 驅動影片: {name}')

print('\n📁 素材就緒:')
!ls -lh /content/examples/my_char/

## 🚀 Step 4: 執行 SCAIL-2 推理

In [ ]:
# @title Run SCAIL-2 inference
%cd /content/SCAIL-2

!python wan/sample_video.py \
  --config configs/scail2_end2end.yaml \
  --checkpoint checkpoints/model/1/fsdp2_rank_0000_checkpoint.pt \
  --ref_image /content/examples/my_char/ref.jpg \
  --driving_video /content/examples/my_char/driving.mp4 \
  --output /content/output.mp4 \
  --resolution 704 \
  --prompt "a character performing the motion, high quality"

print('✅ Done!')

## 📺 Step 5: 預覽結果

In [ ]:
# @title Display output video
from IPython.display import Video
Video('/content/output.mp4', embed=True, width=640)

In [ ]:
# @title Download
from google.colab import files
files.download('/content/output.mp4')